<a href="https://colab.research.google.com/github/YuriArduino/Estudos_Pandas/blob/Pandas_Transforma%C3%A7%C3%A3o_Manipula%C3%A7%C3%A3o_Dados/Projeto_Final_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from typing import Optional, List
import re
from pathlib import Path


class DataLoader:
    """
    Classe para carregar e processar dados JSON de URLs.
    """

    def __init__(self):
        self.dados_cache = {}

    def carregar_dados(
        self,
        url: str,
        chave: str,
        coluna_valor: str,
        prefixos_remover: Optional[List[str]] = None
    ) -> pd.DataFrame:
        """
        Lê um JSON de uma URL, normaliza os dados, explode listas e limpa valores monetários.

        Args:
            url: Link do arquivo JSON
            chave: Nome da chave principal do JSON
            coluna_valor: Nome da coluna com valores monetários
            prefixos_remover: Strings a serem removidas antes da conversão para float

        Returns:
            DataFrame normalizado com valores em float

        Raises:
            ValueError: Se a chave não existir no JSON
            KeyError: Se a coluna_valor não existir no DataFrame
        """
        try:
            # Carrega dados do JSON
            dados_json = pd.read_json(url)

            # Verifica se a chave existe
            if chave not in dados_json:
                raise ValueError(f"Chave '{chave}' não encontrada no JSON")

            # Normaliza os dados
            dados = pd.json_normalize(dados_json[chave])

            # Explode colunas de lista (exceto a primeira)
            colunas_para_explodir = dados.columns[1:].tolist()
            if colunas_para_explodir:
                dados = dados.explode(colunas_para_explodir).reset_index(drop=True)

            # Limpa e converte coluna de valores
            dados = self._limpar_coluna_valores(dados, coluna_valor, prefixos_remover)

            return dados

        except Exception as e:
            raise RuntimeError(f"Erro ao carregar dados de {url}: {str(e)}")

    def _limpar_coluna_valores(
        self,
        dados: pd.DataFrame,
        coluna_valor: str,
        prefixos_remover: Optional[List[str]] = None
    ) -> pd.DataFrame:
        """
        Limpa e converte coluna de valores monetários para float.

        Args:
            dados: DataFrame com os dados
            coluna_valor: Nome da coluna a ser limpa
            prefixos_remover: Prefixos a serem removidos

        Returns:
            DataFrame com coluna limpa
        """
        if coluna_valor not in dados.columns:
            raise KeyError(f"Coluna '{coluna_valor}' não encontrada no DataFrame")

        # Remove prefixos especificados
        if prefixos_remover:
            for prefixo in prefixos_remover:
                dados[coluna_valor] = dados[coluna_valor].str.replace(
                    prefixo, '', regex=False
                )

        # Padroniza separador decimal e remove espaços
        dados[coluna_valor] = (
            dados[coluna_valor]
            .str.replace(',', '.', regex=False)
            .str.strip()
        )

        # Converte para float, tratando erros
        try:
            dados[coluna_valor] = pd.to_numeric(dados[coluna_valor], errors='coerce')
        except Exception:
            # Fallback para conversão manual se necessário
            dados[coluna_valor] = dados[coluna_valor].astype(np.float64)

        return dados

    @staticmethod
    def limpar_texto_cliente(serie: pd.Series) -> pd.Series:
        """
        Limpa e padroniza nomes de clientes.

        Args:
            serie: Série pandas com nomes de clientes

        Returns:
            Série com nomes limpos e padronizados
        """
        return (
            serie
            .str.lower()
            .str.replace(r'[^a-z\s]', ' ', regex=True)
            .str.replace(r'\s+', ' ', regex=True)  # Remove espaços múltiplos
            .str.strip()
        )

    @staticmethod
    def limpar_texto_apartamento(serie: pd.Series, texto_remover: str = '(blocoAP)') -> pd.Series:
        """
        Limpa texto de apartamentos removendo padrões específicos.

        Args:
            serie: Série pandas com dados de apartamentos
            texto_remover: Texto a ser removido

        Returns:
            Série com texto limpo
        """
        return serie.str.replace(texto_remover, '', regex=False).str.strip()

    @staticmethod
    def processar_datas(df: pd.DataFrame, colunas_data: List[str], formato: str = '%d/%m/%Y') -> pd.DataFrame:
        """
        Converte colunas para datetime com formato específico.

        Args:
            df: DataFrame com as colunas de data
            colunas_data: Lista com nomes das colunas de data
            formato: Formato da data (padrão: '%d/%m/%Y')

        Returns:
            DataFrame com colunas convertidas para datetime
        """
        df_copy = df.copy()

        for coluna in colunas_data:
            if coluna in df_copy.columns:
                try:
                    df_copy[coluna] = pd.to_datetime(df_copy[coluna], format=formato)
                except Exception as e:
                    print(f"Erro ao converter coluna '{coluna}': {e}")
                    # Tenta conversão automática como fallback
                    df_copy[coluna] = pd.to_datetime(df_copy[coluna], errors='coerce')

        return df_copy


class AnalisadorVendas:
    """
    Classe para análise de dados de vendas - Projeto 1.
    Identifica clientes com maiores compras durante evento de vendas.
    """

    def __init__(self, dados_vendas: pd.DataFrame):
        self.dados_vendas = dados_vendas

    def calcular_total_compras_por_cliente(self) -> pd.Series:
        """
        Calcula o total de compras por cliente.

        Returns:
            Série com total gasto por cliente (ordenada decrescentemente)
        """
        return (
            self.dados_vendas
            .groupby('Cliente')['Valor da compra']
            .sum()
            .sort_values(ascending=False)
        )

    def identificar_cliente_vencedor(self) -> tuple[str, float]:
        """
        Identifica o cliente com maior valor total de compras.

        Returns:
            Tupla com (nome_cliente, valor_total)
        """
        total_compras = self.calcular_total_compras_por_cliente()
        cliente_vencedor = total_compras.index[0]
        valor_vencedor = total_compras.iloc[0]

        return cliente_vencedor, valor_vencedor

    def gerar_relatorio_vendas(self) -> dict:
        """
        Gera relatório completo das vendas do evento.

        Returns:
            Dicionário com estatísticas das vendas
        """
        total_compras = self.calcular_total_compras_por_cliente()
        cliente_vencedor, valor_vencedor = self.identificar_cliente_vencedor()

        return {
            'periodo_evento': {
                'data_inicio': self.dados_vendas['Data de venda'].min(),
                'data_fim': self.dados_vendas['Data de venda'].max(),
                'duracao_dias': (self.dados_vendas['Data de venda'].max() -
                               self.dados_vendas['Data de venda'].min()).days + 1
            },
            'estatisticas_gerais': {
                'total_clientes': len(total_compras),
                'valor_total_evento': total_compras.sum(),
                'valor_medio_por_cliente': total_compras.mean(),
                'valor_mediano_por_cliente': total_compras.median()
            },
            'cliente_vencedor': {
                'nome': cliente_vencedor,
                'valor_total': valor_vencedor,
                'percentual_do_total': (valor_vencedor / total_compras.sum()) * 100
            },
            'top_5_clientes': total_compras.head(5).to_dict()
        }


class AnalisadorAluguel:
    """
    Classe para análise de atrasos em pagamentos de aluguel - Projeto 2.
    Analisa pontualidade dos pagamentos em condomínio.
    """

    def __init__(self, dados_locacao: pd.DataFrame):
        self.dados_locacao = dados_locacao

    def calcular_atrasos(self) -> pd.DataFrame:
        """
        Calcula os dias de atraso nos pagamentos.

        Returns:
            DataFrame com coluna 'atraso' adicionada
        """
        df_copy = self.dados_locacao.copy()

        # Calcula diferença em dias
        df_copy['atraso'] = (
            df_copy['datas_de_pagamento'] - df_copy['datas_combinadas_pagamento']
        ).dt.days

        return df_copy

    def calcular_media_atraso_por_apartamento(self) -> pd.Series:
        """
        Calcula média de atraso por apartamento.

        Returns:
            Série com média de atraso por apartamento (ordenada decrescentemente)
        """
        dados_com_atraso = self.calcular_atrasos()

        return (
            dados_com_atraso
            .groupby('apartamento')['atraso']
            .mean()
            .sort_values(ascending=False)
        )

    def identificar_situacao_pagamentos(self) -> dict:
        """
        Classifica apartamentos por situação de pagamento.

        Returns:
            Dicionário com apartamentos categorizados por pontualidade
        """
        media_atraso = self.calcular_media_atraso_por_apartamento()

        return {
            'pontuais': media_atraso[media_atraso <= 0].index.tolist(),
            'atraso_leve': media_atraso[(media_atraso > 0) & (media_atraso <= 5)].index.tolist(),
            'atraso_moderado': media_atraso[(media_atraso > 5) & (media_atraso <= 15)].index.tolist(),
            'atraso_severo': media_atraso[media_atraso > 15].index.tolist()
        }

    def gerar_relatorio_alugueis(self) -> dict:
        """
        Gera relatório completo dos atrasos de pagamento.

        Returns:
            Dicionário com estatísticas dos aluguéis
        """
        dados_com_atraso = self.calcular_atrasos()
        media_atraso = self.calcular_media_atraso_por_apartamento()
        situacao_pagamentos = self.identificar_situacao_pagamentos()

        return {
            'estatisticas_gerais': {
                'total_apartamentos': len(media_atraso),
                'atraso_medio_geral': dados_com_atraso['atraso'].mean(),
                'atraso_mediano': dados_com_atraso['atraso'].median(),
                'maior_atraso': dados_com_atraso['atraso'].max(),
                'menor_atraso': dados_com_atraso['atraso'].min()
            },
            'apartamento_mais_atrasado': {
                'apartamento': media_atraso.index[0],
                'atraso_medio': media_atraso.iloc[0]
            },
            'apartamento_mais_pontual': {
                'apartamento': media_atraso.index[-1],
                'atraso_medio': media_atraso.iloc[-1]
            },
            'distribuicao_pontualidade': {
                'pontuais': len(situacao_pagamentos['pontuais']),
                'atraso_leve': len(situacao_pagamentos['atraso_leve']),
                'atraso_moderado': len(situacao_pagamentos['atraso_moderado']),
                'atraso_severo': len(situacao_pagamentos['atraso_severo'])
            },
            'ranking_atrasos': media_atraso.head(10).to_dict()
        }


def main():
    """Função principal para demonstrar o uso das classes e análises."""

    # URLs dos dados
    URLS = {
        'vendas': "https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dados_vendas_clientes.json",
        'locacao': "https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dados_locacao_imoveis.json"
    }

    # Configurações para cada dataset
    CONFIG_DATASETS = {
        'vendas': {
            'chave': 'dados_vendas',
            'coluna_valor': 'Valor da compra',
            'prefixos_remover': ['R$ ']
        },
        'locacao': {
            'chave': 'dados_locacao',
            'coluna_valor': 'valor_aluguel',
            'prefixos_remover': ['$', ' reais']
        }
    }

    # Instancia o carregador
    loader = DataLoader()

    try:

        # Carrega dados de vendas
        dados_vendas = loader.carregar_dados(
            url=URLS['vendas'],
            **CONFIG_DATASETS['vendas']
        )

        # Carrega dados de locação
        dados_locacao = loader.carregar_dados(
            url=URLS['locacao'],
            **CONFIG_DATASETS['locacao']
        )

        # ========== PROCESSAMENTO DADOS DE VENDAS ==========

        # Limpa nomes dos clientes
        if 'Cliente' in dados_vendas.columns:
            dados_vendas['Cliente'] = loader.limpar_texto_cliente(dados_vendas['Cliente'])

        # Converte data de venda
        dados_vendas = loader.processar_datas(dados_vendas, ['Data de venda'])

        # Análise de vendas
        analisador_vendas = AnalisadorVendas(dados_vendas)
        total_compras = analisador_vendas.calcular_total_compras_por_cliente()
        relatorio_vendas = analisador_vendas.gerar_relatorio_vendas()

        # ========== PROCESSAMENTO DADOS DE LOCAÇÃO ==========

        # Limpa apartamentos
        if 'apartamento' in dados_locacao.columns:
            dados_locacao['apartamento'] = loader.limpar_texto_apartamento(
                dados_locacao['apartamento']
            )

        # Converte datas de pagamento
        colunas_data_locacao = ['datas_combinadas_pagamento', 'datas_de_pagamento']
        dados_locacao = loader.processar_datas(dados_locacao, colunas_data_locacao)

        # Análise de aluguéis
        analisador_aluguel = AnalisadorAluguel(dados_locacao)
        dados_locacao = analisador_aluguel.calcular_atrasos()
        media_atraso = analisador_aluguel.calcular_media_atraso_por_apartamento()
        relatorio_alugueis = analisador_aluguel.gerar_relatorio_alugueis()

        # ========== EXIBIÇÃO DOS RESULTADOS ==========

        print("\n" + "="*60)
        print("🎯 PROJETO 1 - ANÁLISE DE VENDAS DO EVENTO")
        print("="*60)

        print(f"📅 Período do evento: {relatorio_vendas['periodo_evento']['data_inicio'].strftime('%d/%m/%Y')} a {relatorio_vendas['periodo_evento']['data_fim'].strftime('%d/%m/%Y')}")
        print(f"⏱️  Duração: {relatorio_vendas['periodo_evento']['duracao_dias']} dias")
        print(f"👥 Total de clientes: {relatorio_vendas['estatisticas_gerais']['total_clientes']}")
        print(f"💰 Valor total do evento: R$ {relatorio_vendas['estatisticas_gerais']['valor_total_evento']:,.2f}")

        print(f"\n🏆 CLIENTE PREMIADO - MAIOR COMPRA DO EVENTO:")
        print(f"   🎉 Parabéns {relatorio_vendas['cliente_vencedor']['nome'].title()}!")
        print(f"   💰 Valor total investido: R$ {relatorio_vendas['cliente_vencedor']['valor_total']:,.2f}")
        print(f"   📊 Representa {relatorio_vendas['cliente_vencedor']['percentual_do_total']:.1f}% de todas as vendas do evento")
        print(f"   🎁 Este cliente receberá o prêmio da loja por ser o maior comprador da semana!")

        print(f"\n📈 ESTRATÉGIAS PARA ATRAIR MAIS CLIENTES:")
        valor_medio = relatorio_vendas['estatisticas_gerais']['valor_medio_por_cliente']
        valor_vencedor = relatorio_vendas['cliente_vencedor']['valor_total']

        if valor_vencedor > valor_medio * 3:
            print(f"   💡 Cliente premiado gastou {valor_vencedor/valor_medio:.1f}x mais que a média!")
            print(f"   🎯 Estratégia: Criar programa VIP para grandes compradores")

        if relatorio_vendas['estatisticas_gerais']['total_clientes'] < 50:
            print(f"   📢 Apenas {relatorio_vendas['estatisticas_gerais']['total_clientes']} clientes participaram do evento")
            print(f"   🎯 Estratégia: Ampliar divulgação e criar campanhas de conscientização")

        print(f"   💰 Ticket médio atual: R$ {valor_medio:.2f}")
        print(f"   🎯 Meta sugerida: Aumentar ticket médio para R$ {valor_medio * 1.2:.2f} (20% de crescimento)")
        print(f"   📊 Potencial de receita adicional: R$ {(valor_medio * 0.2) * relatorio_vendas['estatisticas_gerais']['total_clientes']:,.2f}")

        print(f"\n📋 TOP 5 CLIENTES ESTRATÉGICOS PARA FIDELIZAÇÃO:")
        for i, (cliente, valor) in enumerate(relatorio_vendas['top_5_clientes'].items(), 1):
            status = "🥇 PREMIADO" if i == 1 else f"💎 VIP #{i}"
            print(f"   {status} - {cliente.title()}: R$ {valor:,.2f}")
            if i == 1:
                print(f"      → Este é o nosso cliente campeão da semana! 🏆")

        print(f"\n📊 RESUMO DOS DADOS DE VENDAS:")
        print(dados_vendas.head())

        print("\n" + "="*60)
        print("🏢 PROJETO 2 - ANÁLISE DE ATRASOS DE ALUGUEL")
        print("="*60)

        print(f"\n🏠 ANÁLISE DE ATRASOS NO CONDOMÍNIO FICTÍCIO:")
        print(f"   📊 {relatorio_alugueis['estatisticas_gerais']['total_apartamentos']} apartamentos monitorados")
        print(f"   ⏰ Atraso médio no condomínio: {relatorio_alugueis['estatisticas_gerais']['atraso_medio_geral']:.1f} dias")

        # Análise da situação financeira
        atraso_medio = relatorio_alugueis['estatisticas_gerais']['atraso_medio_geral']
        if atraso_medio > 10:
            print(f"   🚨 ALERTA: Atraso médio de {atraso_medio:.1f} dias indica problemas na gestão financeira!")
        elif atraso_medio > 5:
            print(f"   ⚠️  ATENÇÃO: Atraso médio de {atraso_medio:.1f} dias requer monitoramento")
        else:
            print(f"   ✅ Situação controlada: Atraso médio de apenas {atraso_medio:.1f} dias")

        print(f"\n🔴 MORADOR MAIS PROBLEMÁTICO:")
        apt_mais_atrasado = relatorio_alugueis['apartamento_mais_atrasado']['apartamento']
        atraso_max = relatorio_alugueis['apartamento_mais_atrasado']['atraso_medio']
        print(f"   🏠 Apartamento: {apt_mais_atrasado}")
        print(f"   ⏱️  Atraso médio: {atraso_max:.1f} dias")
        print(f"   💰 Este morador compromete a saúde financeira do condomínio!")

        if atraso_max > 30:
            print(f"   ⚖️  RECOMENDAÇÃO: Considerar ação judicial (atraso > 30 dias)")
        elif atraso_max > 15:
            print(f"   📋 RECOMENDAÇÃO: Notificação formal e cobrança intensiva")
        else:
            print(f"   📞 RECOMENDAÇÃO: Contato direto para regularização")

        print(f"\n🟢 MORADOR MAIS PONTUAL (EXEMPLO PARA OS DEMAIS):")
        apt_mais_pontual = relatorio_alugueis['apartamento_mais_pontual']['apartamento']
        atraso_min = relatorio_alugueis['apartamento_mais_pontual']['atraso_medio']
        print(f"   🏠 Apartamento: {apt_mais_pontual}")
        print(f"   ⏱️  Atraso médio: {atraso_min:.1f} dias")
        if atraso_min <= 0:
            print(f"   ⭐ EXCELENTE: Este morador paga sempre em dia ou antecipado!")
        else:
            print(f"   👍 BOM: Este morador é pontual com raros atrasos")

        print(f"\n📊 IMPACTO FINANCEIRO DOS ATRASOS:")
        dist = relatorio_alugueis['distribuicao_pontualidade']
        total_apts = sum(dist.values())

        print(f"   ✅ Pontuais (≤0 dias): {dist['pontuais']} apartamentos ({dist['pontuais']/total_apts*100:.1f}%)")
        print(f"   🟡 Atraso leve (1-5 dias): {dist['atraso_leve']} apartamentos ({dist['atraso_leve']/total_apts*100:.1f}%)")
        print(f"   🟠 Atraso moderado (6-15 dias): {dist['atraso_moderado']} apartamentos ({dist['atraso_moderado']/total_apts*100:.1f}%)")
        print(f"   🔴 Atraso severo (>15 dias): {dist['atraso_severo']} apartamentos ({dist['atraso_severo']/total_apts*100:.1f}%)")

        # Análise de risco financeiro
        moradores_problematicos = dist['atraso_moderado'] + dist['atraso_severo']
        if moradores_problematicos > total_apts * 0.3:
            print(f"   🚨 CRÍTICO: {moradores_problematicos}/{total_apts} moradores ({moradores_problematicos/total_apts*100:.1f}%) com atraso significativo!")
            print(f"   💸 RISCO: Fluxo de caixa do condomínio pode estar comprometido")
        elif moradores_problematicos > total_apts * 0.15:
            print(f"   ⚠️  ATENÇÃO: {moradores_problematicos}/{total_apts} moradores com atraso preocupante")
        else:
            print(f"   ✅ SAUDÁVEL: Apenas {moradores_problematicos}/{total_apts} moradores com atraso significativo")

        print(f"\n🎯 RECOMENDAÇÕES PARA ADMINISTRAÇÃO:")
        if dist['atraso_severo'] > 0:
            print(f"   📋 Priorizar cobrança dos {dist['atraso_severo']} apartamentos com atraso severo")
        if dist['atraso_moderado'] > 0:
            print(f"   📞 Contatar {dist['atraso_moderado']} moradores com atraso moderado")
        if dist['pontuais'] > total_apts * 0.7:
            print(f"   🏆 Parabenizar os {dist['pontuais']} moradores pontuais (exemplo para os demais)")

        print(f"\n📋 RANKING DOS 10 APARTAMENTOS MAIS ATRASADOS (PRIORIDADE DE COBRANÇA):")
        for i, (apartamento, atraso) in enumerate(relatorio_alugueis['ranking_atrasos'].items(), 1):
            if i <= 3:
                status = f"🔥 URGENTE"
            elif i <= 6:
                status = f"⚠️  ALTA"
            else:
                status = f"📋 MÉDIA"
            print(f"   {i:2d}º {status} - {apartamento}: {atraso:.1f} dias de atraso médio")

            if i == 1 and atraso > 20:
                print(f"      → Este apartamento está causando prejuízo significativo ao condomínio! 💸")

        print(f"\n📊 RESUMO DOS DADOS DE LOCAÇÃO:")
        print(dados_locacao[['apartamento', 'valor_aluguel', 'datas_combinadas_pagamento',
                           'datas_de_pagamento', 'atraso']].head())

        print("\n" + "="*60)
        print("🎯 RESUMO EXECUTIVO DOS PROJETOS")
        print("="*60)

        # Resumo Projeto 1
        cliente_vencedor = relatorio_vendas['cliente_vencedor']['nome'].title()
        valor_premio = relatorio_vendas['cliente_vencedor']['valor_total']
        print(f"\n🏆 PROJETO 1 - EVENTO DE VENDAS:")
        print(f"   ✅ MISSÃO CUMPRIDA: Cliente {cliente_vencedor} identificado como maior comprador!")
        print(f"   🎁 PRÊMIO DEFINIDO: R$ {valor_premio:,.2f} em compras garantem o prêmio da loja")
        print(f"   📈 ESTRATÉGIA FUTURA: {relatorio_vendas['estatisticas_gerais']['total_clientes']} clientes mapeados para novas campanhas")

        # Resumo Projeto 2
        apts_problematicos = relatorio_alugueis['distribuicao_pontualidade']['atraso_severo']
        apt_prioritario = relatorio_alugueis['apartamento_mais_atrasado']['apartamento']
        print(f"\n🏠 PROJETO 2 - GESTÃO DE CONDOMÍNIO:")
        print(f"   ✅ ANÁLISE COMPLETA: Atrasos de todos os moradores mapeados e categorizados")
        print(f"   🎯 AÇÃO PRIORITÁRIA: Apartamento {apt_prioritario} requer atenção imediata")
        print(f"   📊 SITUAÇÃO GERAL: {apts_problematicos} apartamentos com atraso severo identificados")

        print(f"\n💡 PRÓXIMOS PASSOS RECOMENDADOS:")
        print(f"   📈 Vendas: Implementar programa de fidelidade baseado no perfil dos top 5 clientes")
        print(f"   🏠 Condomínio: Iniciar processo de cobrança intensiva dos moradores em atraso severo")
        print(f"   🎯 Ambos: Utilizar estes insights para otimizar processos e aumentar receitas")

    except Exception as e:
        print(f"❌ Erro durante o processamento: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()


🎯 PROJETO 1 - ANÁLISE DE VENDAS DO EVENTO
📅 Período do evento: 06/06/2022 a 10/06/2022
⏱️  Duração: 5 dias
👥 Total de clientes: 9
💰 Valor total do evento: R$ 10,856.55

🏆 CLIENTE PREMIADO - MAIOR COMPRA DO EVENTO:
   🎉 Parabéns Isabely Joanes!
   💰 Valor total investido: R$ 2,329.30
   📊 Representa 21.5% de todas as vendas do evento
   🎁 Este cliente receberá o prêmio da loja por ser o maior comprador da semana!

📈 ESTRATÉGIAS PARA ATRAIR MAIS CLIENTES:
   📢 Apenas 9 clientes participaram do evento
   🎯 Estratégia: Ampliar divulgação e criar campanhas de conscientização
   💰 Ticket médio atual: R$ 1206.28
   🎯 Meta sugerida: Aumentar ticket médio para R$ 1447.54 (20% de crescimento)
   📊 Potencial de receita adicional: R$ 2,171.31

📋 TOP 5 CLIENTES ESTRATÉGICOS PARA FIDELIZAÇÃO:
   🥇 PREMIADO - Isabely Joanes: R$ 2,329.30
      → Este é o nosso cliente campeão da semana! 🏆
   💎 VIP #2 - Maria Julia: R$ 2,086.65
   💎 VIP #3 - Julya Meireles: R$ 1,643.74
   💎 VIP #4 - Diego Armandiu: R$